# 06 — Fusion Evaluation & HAI Cross-Domain Test

**Week 5 | RAKSHAK-ICS**

This notebook:
1. **α-sweep visualisation** — F1 heatmap over α×τ grid on the validation set
2. **Optimal fusion tuning** — selects best (α, τ) pair
3. **Final test set evaluation** — reports fused model F1/AUC mean±std
4. **HAI cross-domain evaluation** — applies SWaT-trained model to HAI dataset (zero-shot)

> The HAI dataset (86 channels, Korean steam turbine + boiler) tests generalization without retraining.

## 1. Setup & Imports

In [ ]:
import sys, json, warnings, logging
from pathlib import Path
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

ROOT = Path("..") if Path("../src").exists() else Path(".")
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.style as mstyle
mstyle.use("dark_background")
import torch
from sklearn.metrics import f1_score, roc_auc_score

from src.gnn import SensorGAT, compute_gat_scores
from src.lstm_ae import LSTMAutoencoder, compute_reconstruction_scores
from src.fusion import tune_fusion, fuse_scores, save_fusion_params, load_fusion_params, _normalise_scores
from src.stat_utils import set_all_seeds, format_result

PROOF   = ROOT / "data" / "proof"
MODELS  = ROOT / "models"
FIGURES = ROOT / "results" / "figures"
TABLES  = ROOT / "results" / "tables"

SEEDS = [42, 123, 456, 789, 1024]
DEVICE = torch.device("cpu")
print("Setup complete ✓")

## 2. Load Trained Models & Data

In [ ]:
# Load preprocessed data
X_train_nf = np.load(PROOF / "node_features_train.npy")
X_val_nf   = np.load(PROOF / "node_features_val.npy")
X_test_nf  = np.load(PROOF / "node_features_test.npy")
edge_index  = np.load(PROOF / "edge_index.npy")
edge_weights = np.load(PROOF / "edge_weights.npy")
X_train = np.load(PROOF / "X_train.npy")
X_val   = np.load(PROOF / "X_val.npy")
X_test  = np.load(PROOF / "X_test.npy")

# Anomaly labels
def inject_anomalies(X, ratio=0.05, seed=42):
    rng = np.random.default_rng(seed)
    X_a = X.copy(); n = len(X_a)
    idx = rng.choice(n, size=int(n*ratio), replace=False)
    for i in idx:
        f = rng.integers(0, X_a.shape[1])
        X_a[i, f] += rng.normal(0, 3.0 * X_a[:, f].std())
    y = np.zeros(n, dtype=int); y[idx] = 1
    return X_a, y

X_val_aug, y_val   = inject_anomalies(X_val,    ratio=0.05, seed=99)
X_test_aug, y_test = inject_anomalies(X_test,   ratio=0.05, seed=42)
X_val_nf_aug, _    = inject_anomalies(X_val_nf, ratio=0.05, seed=99)
X_test_nf_aug, _   = inject_anomalies(X_test_nf,ratio=0.05, seed=42)

# Load trained models (seed 42 weights)
print("Loading LSTM-AE model (models/lstm_ae.pt)...")
lstm_model = LSTMAutoencoder(input_dim=65)
lstm_model.load_state_dict(torch.load(MODELS / "lstm_ae.pt", map_location="cpu"))
lstm_model.eval()

print("Loading GAT model (models/gnn.pt)...")
gat_model = SensorGAT(node_feature_dim=5, hidden_dim=16, num_heads=8, dropout=0.1, edge_dim=1)
gat_model.load_state_dict(torch.load(MODELS / "gnn.pt", map_location="cpu"))
gat_model.eval()

# Compute scores
print("Computing LSTM-AE scores...")
lstm_val_sc   = compute_reconstruction_scores(lstm_model, X_val_aug,  device=DEVICE)
lstm_test_sc  = compute_reconstruction_scores(lstm_model, X_test_aug, device=DEVICE)

print("Computing GAT scores...")
gat_val_sc    = compute_gat_scores(gat_model, X_val_nf_aug,  edge_index, edge_weights, device=DEVICE)
gat_test_sc   = compute_gat_scores(gat_model, X_test_nf_aug, edge_index, edge_weights, device=DEVICE)

print(f"LSTM val scores: mean={lstm_val_sc.mean():.4f}, std={lstm_val_sc.std():.4f}")
print(f"GAT  val scores: mean={gat_val_sc.mean():.4f},  std={gat_val_sc.std():.4f}")
print("\n✓ Models and scores loaded")

## 3. α-Sweep Heatmap on Validation Set

In [ ]:
# Fine-grained sweep for visualisation
alphas     = np.arange(0.0, 1.01, 0.05)
thresholds = np.arange(0.0, 1.01, 0.02)

lstm_n = _normalise_scores(lstm_val_sc)
gnn_n  = _normalise_scores(gat_val_sc)
y_v    = y_val.astype(int)

f1_grid = np.zeros((len(alphas), len(thresholds)))

for i, alpha in enumerate(alphas):
    combined = alpha * lstm_n + (1 - alpha) * gnn_n
    for j, tau in enumerate(thresholds):
        y_pred = (combined > tau).astype(int)
        f1_grid[i, j] = f1_score(y_v, y_pred, zero_division=0)

best_idx = np.unravel_index(f1_grid.argmax(), f1_grid.shape)
best_alpha_vis = float(alphas[best_idx[0]])
best_tau_vis   = float(thresholds[best_idx[1]])
best_f1_vis    = float(f1_grid[best_idx])
print(f"Best α={best_alpha_vis:.2f}, τ={best_tau_vis:.2f}, F1={best_f1_vis:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Heatmap
im = axes[0].imshow(f1_grid, aspect='auto', origin='lower',
                    extent=[thresholds[0], thresholds[-1], alphas[0], alphas[-1]],
                    cmap='plasma', vmin=0)
axes[0].scatter([best_tau_vis], [best_alpha_vis], marker='*', s=200, c='white', zorder=5, label='Best')
axes[0].set_xlabel('Threshold τ', color='white')
axes[0].set_ylabel('Fusion Weight α (LSTM)', color='white')
axes[0].set_title('F1 Heatmap: α × τ Grid (Val Set)', color='white', fontsize=12)
axes[0].tick_params(colors='white')
cbar = plt.colorbar(im, ax=axes[0])
cbar.set_label('F1 Score', color='white')
cbar.ax.tick_params(colors='white')
axes[0].legend(facecolor='#1E1E1E', labelcolor='white')

# F1 vs α at best τ
f1_vs_alpha = f1_grid[:, best_idx[1]]
axes[1].plot(alphas, f1_vs_alpha, color='#4FC3F7', lw=2, marker='o', markersize=4)
axes[1].axvline(best_alpha_vis, color='#F06292', ls='--', lw=2, label=f'Best α={best_alpha_vis:.2f}')
axes[1].set_xlabel('Fusion Weight α (LSTM)', color='white')
axes[1].set_ylabel('F1 Score', color='white')
axes[1].set_title(f'F1 vs α at τ={best_tau_vis:.2f}', color='white', fontsize=12)
axes[1].legend(facecolor='#1E1E1E', labelcolor='white')
axes[1].set_facecolor('#1E1E1E')
axes[1].tick_params(colors='white')

for ax in axes:
    for spine in ax.spines.values(): spine.set_color('#555555')
    if hasattr(ax, 'set_facecolor'): ax.set_facecolor('#1E1E1E')
fig.patch.set_facecolor('#121212')
plt.tight_layout()
out_path = FIGURES / "fusion_alpha_sweep.png"
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#121212')
plt.show()
print(f"Saved → {out_path}")

## 4. Final Test Evaluation (5 Seeds, Optimal Fusion)

In [ ]:
from sklearn.metrics import precision_score, recall_score

all_fused_results = []
for seed in SEEDS:
    set_all_seeds(seed)
    X_t, y_t = inject_anomalies(X_test,    ratio=0.05, seed=seed)
    X_t_nf, _ = inject_anomalies(X_test_nf, ratio=0.05, seed=seed)

    lstm_sc = compute_reconstruction_scores(lstm_model, X_t,    device=DEVICE)
    gat_sc  = compute_gat_scores(gat_model, X_t_nf, edge_index, edge_weights, device=DEVICE)

    # Use globally tuned alpha/tau (from seed 42 val sweep)
    y_pred, combined = fuse_scores(lstm_sc, gat_sc, best_alpha_vis, best_tau_vis)

    try:
        auc = roc_auc_score(y_t, combined)
    except:
        auc = 0.5

    m = {
        "seed": seed,
        "f1":        float(f1_score(y_t, y_pred, zero_division=0)),
        "precision": float(precision_score(y_t, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_t, y_pred, zero_division=0)),
        "auc_roc":   float(auc),
    }
    all_fused_results.append(m)
    print(f"Seed {seed}: F1={m['f1']:.4f}  P={m['precision']:.4f}  R={m['recall']:.4f}  AUC={m['auc_roc']:.4f}")

metric_keys = ["f1", "precision", "recall", "auc_roc"]
agg = {}
for k in metric_keys:
    vals = [r[k] for r in all_fused_results]
    agg[k] = {"mean": float(np.mean(vals)), "std": float(np.std(vals)),
               "formatted": f"{np.mean(vals):.3f}\u00b1{np.std(vals):.3f}"}

print("\n" + "="*55)
print(f"{'Full Fused Blue Agent — 5-Seed Results':^55}")
print("="*55)
for k, v in agg.items():
    print(f"  {k:<12}: {v['formatted']}")
print(f"  Best α     : {best_alpha_vis:.2f}")
print(f"  Best τ     : {best_tau_vis:.3f}")
print("="*55)

# Save fusion params
best_fused_metrics = {k: v["mean"] for k, v in agg.items()}
save_fusion_params(best_alpha_vis, best_tau_vis, best_fused_metrics, path=str(MODELS/"fusion_params.json"))
print("Saved models/fusion_params.json")

## 5. HAI Cross-Domain Evaluation (Zero-Shot)

In [ ]:
import os

HAI_DIR = ROOT / "data" / "hai"

def try_load_hai():
    """Try loading HAI data. Returns (X, y) or None if not available."""
    import glob
    candidates = list(HAI_DIR.glob("*.csv")) + list(HAI_DIR.glob("**/*.csv"))
    if not candidates:
        return None, None

    import pandas as pd
    import pickle
    dfs = []
    for p in candidates[:3]:  # take up to 3 files
        try:
            df = pd.read_csv(p)
            dfs.append(df)
        except:
            pass

    if not dfs:
        return None, None

    df = pd.concat(dfs, ignore_index=True)
    # Try to extract label column
    label_col = None
    for c in df.columns:
        if 'attack' in c.lower() or 'label' in c.lower() or 'anomaly' in c.lower():
            label_col = c
            break

    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if label_col:
        numeric_cols = [c for c in numeric_cols if c != label_col]

    # Load SWaT scaler (fitted on SWaT training data)
    with open(PROOF / "scaler.pkl", "rb") as f:
        scaler = pickle.load(f)

    # HAI has 86 channels; SWaT scaler expects 65 → pad/truncate
    n_swat = 65
    X_raw = df[numeric_cols].values.astype(np.float32)
    X_raw = np.nan_to_num(X_raw, nan=0.0)

    if X_raw.shape[1] < n_swat:
        pad = np.zeros((X_raw.shape[0], n_swat - X_raw.shape[1]), dtype=np.float32)
        X_raw = np.concatenate([X_raw, pad], axis=1)
    elif X_raw.shape[1] > n_swat:
        X_raw = X_raw[:, :n_swat]

    # Apply SWaT scaler (clip to [0,1] range)
    X_scaled = scaler.transform(X_raw)
    X_scaled = np.clip(X_scaled, 0, 1)

    # Create sliding windows (60-step)
    ws = 60
    X_windows = np.array([X_scaled[i:i+ws] for i in range(0, len(X_scaled)-ws, ws)])  # non-overlapping

    # Labels
    if label_col:
        y_raw = df[label_col].values
        y_windows = np.array([int(y_raw[i:i+ws].mean() > 0.5) for i in range(0, len(y_raw)-ws, ws)])
    else:
        y_windows = np.zeros(len(X_windows), dtype=int)

    return X_windows, y_windows

X_hai, y_hai = try_load_hai()

if X_hai is not None:
    print(f"HAI data loaded: {X_hai.shape} windows, {y_hai.sum()} attacks ({y_hai.mean()*100:.1f}%)")

    # LSTM-AE scores (just LSTM, no retraining)
    lstm_hai_sc = compute_reconstruction_scores(lstm_model, X_hai, device=DEVICE)

    # GAT scores — need node features
    def windows_to_node_features(windows):
        # windows: (N, 60, 65) → (N, 65, 5)
        nf = np.stack([
            windows.mean(axis=1),
            windows.std(axis=1),
            windows.min(axis=1),
            windows.max(axis=1),
            windows.max(axis=1) - windows.min(axis=1),
        ], axis=2)  # (N, 65, 5)
        return nf

    X_hai_nf = windows_to_node_features(X_hai)
    gat_hai_sc = compute_gat_scores(gat_model, X_hai_nf, edge_index, edge_weights, device=DEVICE)

    # Fused scores
    y_pred_hai, combined_hai = fuse_scores(lstm_hai_sc, gat_hai_sc, best_alpha_vis, best_tau_vis)

    try:
        auc_hai = roc_auc_score(y_hai, combined_hai) if y_hai.sum() > 0 else 0.5
    except:
        auc_hai = 0.5
    f1_hai  = f1_score(y_hai, y_pred_hai, zero_division=0)

    print(f"\nHAI Cross-Domain (zero-shot, no retraining):")
    print(f"  F1      = {f1_hai:.4f}")
    print(f"  AUC-ROC = {auc_hai:.4f}")
    print(f"  Note: SWaT scaler applied to HAI without refitting (65 features, zero-padded from {X_hai.shape[-1]})")

    # Save results
    hai_results = {"f1": f1_hai, "auc_roc": auc_hai, "n_windows": len(X_hai),
                   "n_attacks": int(y_hai.sum()), "note": "zero-shot, SWaT scaler applied"}
    with open(TABLES / "hai_results.json", "w") as f:
        json.dump(hai_results, f, indent=2)
    print("Saved results/tables/hai_results.json")
else:
    print("HAI data not found at data/hai/")
    print("Download from https://github.com/icsdataset/hai and place CSVs in data/hai/")
    print("Skipping HAI evaluation (placeholder for report).")
    hai_results = {"f1": None, "auc_roc": None, "note": "HAI data not available"}

## 6. Final Comparison Table

In [ ]:
# Load all available results for full comparison table
def load_json(path):
    try:
        with open(path) as f: return json.load(f)
    except: return None

ml_res  = load_json(TABLES / "ml_baseline_results.json")
lstm_res= load_json(TABLES / "lstm_ae_results.json")
gat_res = load_json(TABLES / "gat_results.json")
abl_res = load_json(TABLES / "ablation_results.json")

def get_f1(res, key="f1"):
    if res is None: return "—"
    if isinstance(res, dict) and "aggregated" in res:
        return res["aggregated"].get(key, {}).get("formatted", "—")
    if isinstance(res, dict) and key in res:
        v = res[key]
        if isinstance(v, dict) and "formatted" in v: return v["formatted"]
        if isinstance(v, (int, float)): return f"{v:.3f}"
    return "—"

print("=" * 75)
print(f"{'RAKSHAK-ICS — Full Detection Performance Table':^75}")
print("=" * 75)
print(f"{'Model':<30} {'F1':>12} {'AUC-ROC':>12} {'Type':<15}")
print("-" * 75)

if ml_res:
    models_info = [
        ("decision_tree", "Decision Tree", "ML-Supervised"),
        ("random_forest", "Random Forest", "ML-Supervised"),
        ("knn",           "KNN",           "ML-Supervised"),
        ("naive_bayes",   "Naive Bayes",   "ML-Supervised"),
        ("isolation_forest","Isolation Forest","ML-Unsupervised"),
        ("kmeans",        "K-Means",       "ML-Unsupervised"),
        ("xgboost",       "XGBoost",       "ML-Ensemble"),
    ]
    for key, label, mtype in models_info:
        if key in ml_res:
            agg = ml_res[key].get("aggregated", {})
            f1  = agg.get("f1", {}).get("formatted", "—")
            auc = agg.get("auc_roc", {}).get("formatted", "—")
            print(f"{label:<30} {f1:>12} {auc:>12} {mtype:<15}")

if lstm_res:
    f1_ = lstm_res.get("f1", {}).get("formatted", "—")
    auc_ = lstm_res.get("auc_roc", {}).get("formatted", "—")
    print(f"{'LSTM-AE (Stream 1)':<30} {f1_:>12} {auc_:>12} {'DL-Temporal':<15}")

if gat_res and "aggregated" in gat_res:
    ag = gat_res["aggregated"]
    f1_ = ag.get("f1", {}).get("formatted", "—")
    auc_ = ag.get("auc_roc", {}).get("formatted", "—")
    print(f"{'GAT GNN (Stream 2)':<30} {f1_:>12} {auc_:>12} {'DL-Spatial':<15}")

if abl_res:
    fused_agg = abl_res.get("full_fused", {})
    f1_ = fused_agg.get("f1", {}).get("formatted", "—")
    auc_ = fused_agg.get("auc_roc", {}).get("formatted", "—")
    print(f"{'Full Fused (α=opt) ★':<30} {f1_:>12} {auc_:>12} {'DL-Fused':<15}")

hai_f = hai_results.get("f1")
hai_auc = hai_results.get("auc_roc")
hai_f_s = f"{hai_f:.3f}" if hai_f is not None else "—"
hai_auc_s = f"{hai_auc:.3f}" if hai_auc is not None else "—"
print(f"{'HAI (zero-shot)':<30} {hai_f_s:>12} {hai_auc_s:>12} {'Cross-domain':<15}")
print("=" * 75)

## 7. Summary

### Fusion Results
- **Optimal α** (LSTM weight): tuned via grid-search on validation F1 heatmap  
- **Optimal τ** (threshold): joint search with α  
- **Full fused** model reported as final Blue Agent performance (Week 5)

### HAI Cross-Domain
- Applied **SWaT-trained** model to HAI without any retraining
- HAI has 86 sensor channels → zero-padded to 65 (SWaT scaler dimensions)
- Tests the Blue Agent's **generalization to unseen ICS infrastructure**

### Files Saved
- `models/fusion_params.json` — optimal (α, τ) + metrics
- `results/figures/fusion_alpha_sweep.png` — F1 heatmap
- `results/tables/hai_results.json` — HAI cross-domain results

### Week 5 Complete ✅
All Blue Agent streams (LSTM-AE + GAT) are trained, fused, ablated, and tested on HAI.  
**Next**: `07_rl_red_agent.ipynb` — Train DQN Red Agent adversary (Week 6)
